# Block 3 — live lecture demo

Instructor notebook for the 15-minute Block 3 lecture. Run top to bottom.

Four validation layers, what each one sees that the others are blind to, and the
repair loop. Your exercise is `validate_check_exercise.ipynb`.

## 0 · Setup

In [ ]:
import os
import odmlib.define_loader as DL
import odmlib.loader as LD
from odmlib.odm_parser import ODMSchemaValidator
from odmlib import create_oid_checker
from odmlib.define_2_1.rules.metadata_schema import MetadataSchema

os.makedirs("output", exist_ok=True)

def load_define(path):
    loader = LD.ODMLoader(DL.XMLDefineLoader(model_package="define_2_1"))
    loader.open_odm_document(path)
    return loader.root()          # one root() call - keep this reference for edits

xsd = ODMSchemaValidator(standard="define", version="2.1")   # schema ships with odmlib
print("ready")

## 1 · All four layers on a known-good file

The official v2.1 schema is bundled — nothing to download, no paths to configure.

In [ ]:
xsd.validate_file("../data/defineV21-SDTM.xml")
print("layer 1 (XSD):         PASS")

odm = load_define("../data/defineV21-SDTM.xml")

odm.verify_oids(create_oid_checker("define_2_1"))
print("layer 2 (OID ref/def): PASS")

odm.verify_conformance(MetadataSchema())
print("layer 3 (conformance): PASS")

odm.verify_order()
print("layer 4 (order):       PASS")

## 2 · The combined call — your default

Runs layers 2–4 and collects **every** finding. Empty list means clean.

In [ ]:
errors = odm.validate(
    collect_errors=True,
    oid_checker=create_oid_checker("define_2_1"),
    conformance_checker=MetadataSchema(),
)
print(f"combined validate: {len(errors)} errors")

## 3 · Why you need *both* XSD and the object checks

**File A** — schema-valid, but an `ItemRef` points at an OID nothing defines.
XSD sees a string; layer 2 sees a dangling edge.

In [ ]:
from odmlib.exceptions import OdmlibOIDError

xsd.validate_file("../data/define_broken_refs.xml")
print("XSD:     PASS\n")

broken = load_define("../data/define_broken_refs.xml")
try:
    broken.verify_oids(create_oid_checker("define_2_1"))
except OdmlibOIDError as e:
    print("layer 2: FAIL ->", e)

**File B** — the exact reverse. `def:Class/@Name` is an XSD enumeration; the object layers have nothing to say about it.

In [ ]:
bad_class = load_define("../data/defineV21-SDTM-invalid-class.xml")

errors = bad_class.validate(
    collect_errors=True,
    oid_checker=create_oid_checker("define_2_1"),
    conformance_checker=MetadataSchema(),
)
print(f"object layers: {len(errors)} errors\n")

print("XSD:")
for err in xsd.xsd.iter_errors("../data/defineV21-SDTM-invalid-class.xml"):
    print("  ", err.reason[:110])

Two files, two failure classes, neither layer catches both. **Run both, every time.**

## 4 · The repair loop

Strict loading *rejects* a malformed file — correct, and useless when your job is
repairing it. `permissive()` gets it into objects so validation can tell you
everything that's wrong.

In [ ]:
import odmlib
from odmlib.exceptions import OdmlibRequiredAttributeError

try:
    load_define("../data/nonconformant_define21.xml")
except OdmlibRequiredAttributeError as e:
    print("strict load fails:", e)

with odmlib.permissive():
    nc = load_define("../data/nonconformant_define21.xml")
print("\npermissive load OK")

In [ ]:
errors = nc.validate(
    collect_errors=True,
    oid_checker=create_oid_checker("define_2_1"),
    conformance_checker=MetadataSchema(),
)
for err in errors:
    print(type(err).__name__, "-", str(err).splitlines()[0])

Fixes are ordinary attribute assignments on the object tree — no text editing:

In [ ]:
nc_mdv = nc.Study.MetaDataVersion

nc_mdv.ItemGroupDef[0].Repeating = "No"        # the missing required attribute
nc_mdv.ItemDef[0].Origin[0].Type = "Collected" # the invalid Origin type

errors = nc.validate(
    collect_errors=True,
    oid_checker=create_oid_checker("define_2_1"),
    conformance_checker=MetadataSchema(),
)
print(f"after repair: {len(errors)} errors")

Load (permissively if needed) → collect every error → fix objects → re-validate to
clean → write. **That loop is the core odmlib workflow.**

---

**Your turn:** `validate_check_exercise.ipynb` — 4 TODOs, ~20 minutes.